<a href="https://colab.research.google.com/github/yhshengjy/ClinPKPD/blob/main/Notebook6_%CE%B2%E5%86%85%E9%85%B0%E8%83%BA%E7%B1%BB_PKPD%E6%A8%A1%E6%8B%9F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 6：β-内酰胺类 PK/PD 模拟：理解 %fT > MIC

本 Notebook 是临床药学 PK/PD 交互式模拟平台的第五个模块。

本节聚焦 β-内酰胺类抗菌药物的核心 PK/PD 指标：

$$
\% fT > MIC
$$

也就是：

> 在一个给药间隔内，游离药物浓度高于病原菌 MIC 的时间百分比。

β-内酰胺类药物通常表现为“时间依赖性杀菌”。因此，与单纯追求很高的峰浓度相比，维持足够长时间的游离药物浓度高于 MIC 通常更加重要。

本 Notebook 将通过一室静脉输注模型，帮助你比较：

- 短时输注
- 延长输注
- 连续输注
- 不同 MIC
- 不同清除率 CL
- 不同蛋白结合率
- 不同 PK/PD 目标

本节中使用的具体药物示例为 **美罗培南 meropenem**。所有参数均用于教学模拟，不可直接作为真实患者处方依据。

## 1. 学习目标

完成本 Notebook 后，你应该能够：

1. 解释为什么 β-内酰胺类抗菌药物主要使用时间依赖性 PK/PD 指标进行评价。
2. 描述 MIC、游离药物浓度和蛋白结合率之间的关系。
3. 计算并解释一个给药间隔内的 $\%fT > MIC$。
4. 比较短时输注、延长输注和持续输注策略。
5. 分析 MIC、清除率、分布容积和输注策略如何影响 PK/PD 目标达成。

## 2. β-内酰胺类抗菌药物的 PK/PD 特点

β-内酰胺类抗菌药物包括：

- 青霉素类
- 头孢菌素类
- 碳青霉烯类
- 单环 β-内酰胺类
- β-内酰胺/β-内酰胺酶抑制剂复方制剂

这类药物的抗菌效应通常与以下指标关系最密切：

$$
\% fT > MIC
$$

其中：

| 符号 | 含义 |
|---|---|
| f | free，游离药物部分 |
| T | time，时间 |
| MIC | minimum inhibitory concentration，最低抑菌浓度 |
| %fT > MIC | 一个给药间隔内，游离药物浓度高于 MIC 的时间百分比 |

只有游离药物浓度通常被认为能够直接发挥抗菌活性。因此，对于蛋白结合率较高的药物，需要关注游离浓度，而不是只看总浓度。

$$
C_{free}(t) = C_{total}(t) \times (1 - Protein\ Binding)
$$

例如，如果总浓度为 10 mg/L，蛋白结合率为 20%，则游离浓度为：

$$
C_{free} = 10 \times (1 - 0.20) = 8\ mg/L
$$

## 3. β-内酰胺类常见 PK/PD 目标

不同 β-内酰胺类药物、不同感染部位、不同患者状态和不同病原菌 MIC，所需 PK/PD 目标可能不同。

教学中常见的简化目标包括：

| 药物类别 | 常见教学目标示例 |
|---|---|
| 青霉素类 | 约 40%–50% fT > MIC |
| 头孢菌素类 | 约 50%–70% fT > MIC |
| 碳青霉烯类 | 约 40% fT > MIC |
| 重症感染或免疫功能低下 | 可能需要更高目标，例如 100% fT > MIC 或 100% fT > 4×MIC |

本 Notebook 默认采用可调目标，让你观察不同目标设定下的达标情况。

注意：

> 真实临床中，PK/PD 目标不是固定不变的。它会受到感染严重程度、感染部位、病原菌 MIC、患者免疫状态、肾功能、组织穿透率和药物安全性等因素影响。

## 4. 美罗培南作为教学示例

本 Notebook 以美罗培南作为 β-内酰胺类药物示例。

美罗培南属于碳青霉烯类抗菌药物，临床上以静脉给药为主。

在官方说明书中，美罗培南可采用静脉输注给药，常规输注时间通常为 15–30 分钟；其血浆蛋白结合率较低，约为 2%。本 Notebook 中默认蛋白结合率设置为 2%，用于说明游离浓度和总浓度之间的关系。

为了便于教学，本 Notebook 采用简化的一室静脉输注模型。模型不代表所有患者，也不能替代说明书、指南、TDM 或临床药师评估。

## 5. 一室静脉输注模型

对于静脉输注给药，如果输注时间为 $T_{inf}$，输注速率为：

$$
R_0 = \frac{Dose}{T_{inf}}
$$

在输注过程中，血药浓度可表示为：

$$
C(t) = \frac{R_0}{CL} \left(1-e^{-kt}\right)
$$

其中：

$$
k = \frac{CL}{V_d}
$$

输注结束后的浓度下降可表示为：

$$
C(t) = C_{end} \cdot e^{-k(t-T_{inf})}
$$

其中：

$$
C_{end} = \frac{R_0}{CL}\left(1-e^{-kT_{inf}}\right)
$$

多剂量给药时，可以把每一次输注产生的浓度贡献相加。

本 Notebook 将模拟：

$$
Dose\ q\tau\ h
$$

例如：

$$
1000\ mg\ q8h
$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


def concentration_single_infusion(t_after_dose, dose_mg, vd_l, cl_l_h, infusion_h):
    """
    Concentration contribution from one IV infusion dose in a one-compartment model.
    """
    k_elim = cl_l_h / vd_l
    infusion_h = max(infusion_h, 1e-6)
    rate_mg_h = dose_mg / infusion_h

    concentration = np.zeros_like(t_after_dose, dtype=float)

    during = (t_after_dose >= 0) & (t_after_dose <= infusion_h)
    after = t_after_dose > infusion_h

    concentration[during] = (
        rate_mg_h / cl_l_h * (1 - np.exp(-k_elim * t_after_dose[during]))
    )

    c_end = rate_mg_h / cl_l_h * (1 - np.exp(-k_elim * infusion_h))
    concentration[after] = c_end * np.exp(-k_elim * (t_after_dose[after] - infusion_h))

    return concentration


def concentration_multiple_infusions(t, dose_mg, vd_l, cl_l_h, tau_h, infusion_h, n_doses):
    """
    Concentration-time profile after repeated intermittent IV infusions.
    """
    concentration = np.zeros_like(t, dtype=float)
    dose_times = np.arange(n_doses) * tau_h

    for dose_time in dose_times:
        t_after_dose = t - dose_time
        concentration += concentration_single_infusion(
            t_after_dose=t_after_dose,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            infusion_h=infusion_h
        )

    return concentration


def concentration_continuous_infusion(t, daily_dose_mg, vd_l, cl_l_h, loading_dose_mg=0):
    """
    Concentration-time profile during continuous infusion with optional loading dose.
    """
    k_elim = cl_l_h / vd_l
    rate_mg_h = daily_dose_mg / 24

    infusion_component = rate_mg_h / cl_l_h * (1 - np.exp(-k_elim * t))
    loading_component = (loading_dose_mg / vd_l) * np.exp(-k_elim * t)

    concentration = infusion_component + loading_component
    return concentration


def free_concentration(total_concentration, protein_binding_percent):
    """
    Convert total concentration to free concentration.
    """
    free_fraction = 1 - protein_binding_percent / 100
    return total_concentration * free_fraction


def percent_time_above_threshold(t, concentration, threshold, start_time, end_time):
    """
    Calculate percentage of time concentration is above a threshold during a time window.
    """
    mask = (t >= start_time) & (t <= end_time)
    if np.sum(mask) < 2:
        return np.nan

    t_window = t[mask]
    c_window = concentration[mask]
    above = (c_window > threshold).astype(float)

    duration = t_window[-1] - t_window[0]
    if duration <= 0:
        return np.nan

    percent = np.trapz(above, t_window) / duration * 100
    return percent


def calculate_interval_metrics(t, total_conc, free_conc, mic, start_time, end_time):
    """
    Calculate PK/PD metrics in a selected dosing interval.
    """
    mask = (t >= start_time) & (t <= end_time)
    t_window = t[mask]
    total_window = total_conc[mask]
    free_window = free_conc[mask]

    ft_mic = percent_time_above_threshold(
        t=t,
        concentration=free_conc,
        threshold=mic,
        start_time=start_time,
        end_time=end_time
    )

    metrics = {
        "Total Cmax": np.max(total_window),
        "Total Cmin": np.min(total_window),
        "Free Cmax": np.max(free_window),
        "Free Cmin": np.min(free_window),
        "%fT>MIC": ft_mic,
        "Interval start": start_time,
        "Interval end": end_time
    }

    return metrics

## 6. 交互模拟 1：多剂量输注与 %fT > MIC

下面的模拟展示多剂量静脉输注后，血药浓度如何随时间变化。

请重点观察：

- 总浓度曲线
- 游离浓度曲线
- MIC 水平线
- 游离浓度高于 MIC 的时间比例
- 给药间隔内是否达到目标 \%fT > MIC

在这个模拟中，你可以调整：

- Dose：每次给药剂量
- Vd：表观分布容积
- CL：清除率
- Tau：给药间隔
- Infusion：输注时间
- Protein binding：蛋白结合率
- MIC：最低抑菌浓度
- Target：预设 PK/PD 目标

In [ ]:
def plot_beta_lactam_pkpd(
    dose_mg=1000,
    vd_l=20,
    cl_l_h=10,
    tau_h=8,
    infusion_h=0.5,
    n_doses=6,
    protein_binding_percent=2,
    mic=2,
    target_percent=40
):
    t_end_h = tau_h * n_doses
    t = np.linspace(0, t_end_h, 2500)

    total_conc = concentration_multiple_infusions(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        infusion_h=infusion_h,
        n_doses=n_doses
    )

    free_conc = free_concentration(total_conc, protein_binding_percent)

    last_start = (n_doses - 1) * tau_h
    last_end = n_doses * tau_h

    metrics = calculate_interval_metrics(
        t=t,
        total_conc=total_conc,
        free_conc=free_conc,
        mic=mic,
        start_time=last_start,
        end_time=last_end
    )

    target_achieved = metrics["%fT>MIC"] >= target_percent

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(t, total_conc, linewidth=2, label="Total concentration")
    ax.plot(t, free_conc, linewidth=2, linestyle="--", label="Free concentration")
    ax.axhline(mic, linestyle=":", linewidth=2, label=f"MIC = {mic:.2f} mg/L")

    ax.fill_between(
        t,
        free_conc,
        mic,
        where=free_conc > mic,
        alpha=0.15,
        interpolate=True,
        label="Free concentration above MIC"
    )

    ax.axvspan(last_start, last_end, alpha=0.08, label="Last dosing interval")

    ax.set_title("Multiple-Dose Beta-Lactam PK/PD Simulation")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend(loc="upper right")
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Dose",
            "Dosing interval",
            "Infusion duration",
            "Vd",
            "CL",
            "Protein binding",
            "MIC",
            "Target %fT>MIC",
            "Last-interval %fT>MIC",
            "Target achieved",
            "Total Cmax in last interval",
            "Total Cmin in last interval",
            "Free Cmax in last interval",
            "Free Cmin in last interval"
        ],
        "Value": [
            f"{dose_mg:.0f} mg",
            f"q{tau_h:.1f}h",
            f"{infusion_h:.2f} h",
            f"{vd_l:.1f} L",
            f"{cl_l_h:.1f} L/h",
            f"{protein_binding_percent:.1f}%",
            f"{mic:.2f} mg/L",
            f"{target_percent:.0f}%",
            f"{metrics['%fT>MIC']:.1f}%",
            "Yes" if target_achieved else "No",
            f"{metrics['Total Cmax']:.2f} mg/L",
            f"{metrics['Total Cmin']:.2f} mg/L",
            f"{metrics['Free Cmax']:.2f} mg/L",
            f"{metrics['Free Cmin']:.2f} mg/L"
        ]
    })

    display(summary)


interact(
    plot_beta_lactam_pkpd,
    dose_mg=FloatSlider(value=1000, min=250, max=3000, step=250, description="Dose"),
    vd_l=FloatSlider(value=20, min=5, max=80, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=10, min=1, max=30, step=1, description="CL"),
    tau_h=FloatSlider(value=8, min=4, max=24, step=2, description="Tau"),
    infusion_h=FloatSlider(value=0.5, min=0.25, max=8, step=0.25, description="Infusion"),
    n_doses=IntSlider(value=6, min=2, max=12, step=1, description="Doses"),
    protein_binding_percent=FloatSlider(value=2, min=0, max=95, step=1, description="Binding"),
    mic=FloatSlider(value=2, min=0.25, max=16, step=0.25, description="MIC"),
    target_percent=FloatSlider(value=40, min=20, max=100, step=5, description="Target")
);

interactive(children=(FloatSlider(value=1000.0, description='Dose', max=3000.0, min=250.0, step=250.0), FloatS…

## 7. 观察任务 1：哪些因素影响 %fT > MIC？

请使用上面的交互模拟完成以下任务。

### 任务 A：标准方案

设置：

- Dose = 1000 mg
- Vd = 20 L
- CL = 10 L/h
- Tau = 8 h
- Infusion = 0.5 h
- Protein binding = 2%
- MIC = 2 mg/L
- Target = 40%

记录：

- Last-interval %fT > MIC
- Total Cmax
- Total Cmin
- Free Cmax
- Free Cmin
- 是否达标

### 任务 B：MIC 升高

只将 MIC 改为 4 mg/L 或 8 mg/L。

观察：

- %fT > MIC 是否下降？
- 原方案是否仍然达标？
- 为什么同一个剂量在不同 MIC 下可能出现不同疗效？

### 任务 C：清除率增加

将 CL 改为 20 L/h。

观察：

- 浓度下降是否更快？
- %fT > MIC 是否下降？
- 这种情况可以模拟什么临床场景？

提示：部分危重症患者可能出现增强肾清除，导致 β-内酰胺类药物浓度低于预期。

### 任务 D：清除率降低

将 CL 改为 4 L/h。

观察：

- 浓度是否维持更久？
- %fT > MIC 是否升高？
- 是否一定意味着越高越好？

提示：CL 降低可能提高达标率，但也可能增加不良反应风险，尤其在肾功能不全患者中需要谨慎。

## 8. 短时输注、延长输注和连续输注

β-内酰胺类药物的核心目标是让游离浓度在足够长时间内高于 MIC。

因此，改变输注方式可能明显影响 \%fT > MIC。

常见给药方式包括：

| 给药方式 | 特点 |
|---|---|
| 短时输注 | 输注时间短，峰浓度较高，但浓度下降较快 |
| 延长输注 | 输注时间延长，峰浓度可能较低，但高于 MIC 的时间可能延长 |
| 连续输注 | 持续给药，目标是维持较稳定浓度 |

需要注意：

> 延长输注或连续输注不是所有情况下都必须使用。是否采用应结合药物稳定性、感染严重程度、MIC、肾功能、护理可行性和医院规范。

In [ ]:
def compare_infusion_strategies(
    dose_mg=1000,
    vd_l=20,
    cl_l_h=10,
    tau_h=8,
    short_infusion_h=0.5,
    extended_infusion_h=3,
    protein_binding_percent=2,
    mic=2,
    target_percent=40,
    loading_dose_mg=0
):
    t_end_h = 48
    t = np.linspace(0, t_end_h, 3000)
    n_doses = int(np.floor(t_end_h / tau_h))
    daily_dose_mg = dose_mg * (24 / tau_h)

    short_total = concentration_multiple_infusions(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        infusion_h=short_infusion_h,
        n_doses=n_doses
    )

    extended_total = concentration_multiple_infusions(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        infusion_h=extended_infusion_h,
        n_doses=n_doses
    )

    continuous_total = concentration_continuous_infusion(
        t=t,
        daily_dose_mg=daily_dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        loading_dose_mg=loading_dose_mg
    )

    short_free = free_concentration(short_total, protein_binding_percent)
    extended_free = free_concentration(extended_total, protein_binding_percent)
    continuous_free = free_concentration(continuous_total, protein_binding_percent)

    last_start = t_end_h - tau_h
    last_end = t_end_h
    last_24_start = t_end_h - 24

    short_metrics = calculate_interval_metrics(t, short_total, short_free, mic, last_start, last_end)
    extended_metrics = calculate_interval_metrics(t, extended_total, extended_free, mic, last_start, last_end)

    continuous_ft = percent_time_above_threshold(
        t=t,
        concentration=continuous_free,
        threshold=mic,
        start_time=last_24_start,
        end_time=t_end_h
    )

    continuous_mask = (t >= last_24_start) & (t <= t_end_h)
    continuous_metrics = {
        "Total Cmax": np.max(continuous_total[continuous_mask]),
        "Total Cmin": np.min(continuous_total[continuous_mask]),
        "Free Cmax": np.max(continuous_free[continuous_mask]),
        "Free Cmin": np.min(continuous_free[continuous_mask]),
        "%fT>MIC": continuous_ft
    }

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(t, short_free, linewidth=2, label=f"Short infusion ({short_infusion_h:.1f} h)")
    ax.plot(t, extended_free, linewidth=2, label=f"Extended infusion ({extended_infusion_h:.1f} h)")
    ax.plot(t, continuous_free, linewidth=2, label="Continuous infusion")
    ax.axhline(mic, linestyle=":", linewidth=2, label=f"MIC = {mic:.2f} mg/L")
    ax.axvspan(last_start, last_end, alpha=0.08, label="Last intermittent interval")

    ax.set_title("Short vs Extended vs Continuous Infusion")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Free concentration (mg/L)")
    ax.legend(loc="upper right")
    plt.show()

    summary = pd.DataFrame({
        "Strategy": ["Short infusion", "Extended infusion", "Continuous infusion"],
        "Infusion duration": [
            f"{short_infusion_h:.2f} h",
            f"{extended_infusion_h:.2f} h",
            "24 h"
        ],
        "Total daily dose": [
            f"{daily_dose_mg:.0f} mg/day",
            f"{daily_dose_mg:.0f} mg/day",
            f"{daily_dose_mg:.0f} mg/day"
        ],
        "%fT>MIC": [
            f"{short_metrics['%fT>MIC']:.1f}%",
            f"{extended_metrics['%fT>MIC']:.1f}%",
            f"{continuous_metrics['%fT>MIC']:.1f}%"
        ],
        "Target achieved": [
            "Yes" if short_metrics["%fT>MIC"] >= target_percent else "No",
            "Yes" if extended_metrics["%fT>MIC"] >= target_percent else "No",
            "Yes" if continuous_metrics["%fT>MIC"] >= target_percent else "No"
        ],
        "Free Cmax": [
            f"{short_metrics['Free Cmax']:.2f} mg/L",
            f"{extended_metrics['Free Cmax']:.2f} mg/L",
            f"{continuous_metrics['Free Cmax']:.2f} mg/L"
        ],
        "Free Cmin": [
            f"{short_metrics['Free Cmin']:.2f} mg/L",
            f"{extended_metrics['Free Cmin']:.2f} mg/L",
            f"{continuous_metrics['Free Cmin']:.2f} mg/L"
        ]
    })

    display(summary)


interact(
    compare_infusion_strategies,
    dose_mg=FloatSlider(value=1000, min=250, max=3000, step=250, description="Dose"),
    vd_l=FloatSlider(value=20, min=5, max=80, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=10, min=1, max=30, step=1, description="CL"),
    tau_h=FloatSlider(value=8, min=4, max=24, step=2, description="Tau"),
    short_infusion_h=FloatSlider(value=0.5, min=0.25, max=2, step=0.25, description="Short"),
    extended_infusion_h=FloatSlider(value=3, min=1, max=8, step=0.5, description="Extended"),
    protein_binding_percent=FloatSlider(value=2, min=0, max=95, step=1, description="Binding"),
    mic=FloatSlider(value=2, min=0.25, max=16, step=0.25, description="MIC"),
    target_percent=FloatSlider(value=40, min=20, max=100, step=5, description="Target"),
    loading_dose_mg=FloatSlider(value=0, min=0, max=3000, step=250, description="Loading")
);

interactive(children=(FloatSlider(value=1000.0, description='Dose', max=3000.0, min=250.0, step=250.0), FloatS…

## 9. 观察任务 2：为什么延长输注可能提高达标率？

请使用上面的比较模拟完成以下任务。

### 任务 A：比较三种输注方式

设置：

- Dose = 1000 mg
- Tau = 8 h
- Short infusion = 0.5 h
- Extended infusion = 3 h
- CL = 10 L/h
- Vd = 20 L
- MIC = 2 mg/L
- Target = 40%

比较：

- 短时输注的 %fT > MIC
- 延长输注的 %fT > MIC
- 连续输注的 %fT > MIC

思考：

> 为什么延长输注的峰浓度可能降低，但 %fT > MIC 反而可能增加？

### 任务 B：MIC 升高

将 MIC 改为 4 mg/L 或 8 mg/L。

观察：

- 哪种输注方式更容易维持浓度高于 MIC？
- 原来的短时输注方案是否可能不达标？

### 任务 C：总日剂量相同

在这个比较中，短时输注、延长输注和连续输注使用相同的总日剂量。

思考：

> 当总日剂量相同时，仅改变输注方式为什么会改变 PK/PD 达标情况？

## 10. MIC 敏感性分析

MIC 是连接药物暴露和抗菌效应的重要参数。

同一个给药方案，在 MIC 较低时可能达标；当 MIC 升高时，可能不再达标。

下面的模拟固定给药方案，然后观察不同 MIC 下的 \%fT > MIC。

In [ ]:
def mic_sensitivity_analysis(
    dose_mg=1000,
    vd_l=20,
    cl_l_h=10,
    tau_h=8,
    infusion_h=3,
    n_doses=6,
    protein_binding_percent=2,
    target_percent=40
):
    mic_values = np.array([0.25, 0.5, 1, 2, 4, 8, 16])
    t_end_h = tau_h * n_doses
    t = np.linspace(0, t_end_h, 2500)

    total_conc = concentration_multiple_infusions(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        infusion_h=infusion_h,
        n_doses=n_doses
    )

    free_conc = free_concentration(total_conc, protein_binding_percent)

    last_start = (n_doses - 1) * tau_h
    last_end = n_doses * tau_h

    ft_values = []
    for mic in mic_values:
        ft = percent_time_above_threshold(
            t=t,
            concentration=free_conc,
            threshold=mic,
            start_time=last_start,
            end_time=last_end
        )
        ft_values.append(ft)

    fig, ax = plt.subplots()
    ax.plot(mic_values, ft_values, marker="o", linewidth=2)
    ax.axhline(target_percent, linestyle="--", label=f"Target = {target_percent:.0f}%")
    ax.set_xscale("log", base=2)
    ax.set_xticks(mic_values)
    ax.set_xticklabels([str(x) for x in mic_values])
    ax.set_title("MIC Sensitivity Analysis")
    ax.set_xlabel("MIC (mg/L)")
    ax.set_ylabel("%fT>MIC in last interval")
    ax.set_ylim(0, 105)
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "MIC (mg/L)": mic_values,
        "%fT>MIC": [f"{x:.1f}%" for x in ft_values],
        "Target achieved": ["Yes" if x >= target_percent else "No" for x in ft_values]
    })

    display(summary)


interact(
    mic_sensitivity_analysis,
    dose_mg=FloatSlider(value=1000, min=250, max=3000, step=250, description="Dose"),
    vd_l=FloatSlider(value=20, min=5, max=80, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=10, min=1, max=30, step=1, description="CL"),
    tau_h=FloatSlider(value=8, min=4, max=24, step=2, description="Tau"),
    infusion_h=FloatSlider(value=3, min=0.25, max=8, step=0.25, description="Infusion"),
    n_doses=IntSlider(value=6, min=2, max=12, step=1, description="Doses"),
    protein_binding_percent=FloatSlider(value=2, min=0, max=95, step=1, description="Binding"),
    target_percent=FloatSlider(value=40, min=20, max=100, step=5, description="Target")
);

interactive(children=(FloatSlider(value=1000.0, description='Dose', max=3000.0, min=250.0, step=250.0), FloatS…

## 11. 观察任务 3：MIC 与治疗成功概率的关系

请使用上面的 MIC 敏感性分析完成以下任务。

### 任务 A：默认方案

设置：

- Dose = 1000 mg
- Tau = 8 h
- Infusion = 3 h
- CL = 10 L/h
- Vd = 20 L
- Protein binding = 2%
- Target = 40%

观察：

- 当 MIC = 1 mg/L 时是否达标？
- 当 MIC = 4 mg/L 时是否达标？
- 当 MIC = 8 mg/L 时是否达标？

### 任务 B：提高目标

将 Target 从 40% 改为 100%。

观察：

- 哪些 MIC 下仍然达标？
- 为什么重症感染或免疫功能低下患者可能需要更高目标？

### 任务 C：改变 CL

将 CL 改为 20 L/h。

观察：

- 同一 MIC 下，%fT > MIC 是否下降？
- 为什么增强清除患者可能需要更积极的剂量优化？

## 12. 蛋白结合率对游离浓度的影响

β-内酰胺类药物的蛋白结合率差异很大。

对于低蛋白结合率药物，总浓度和游离浓度相差不大。  
对于高蛋白结合率药物，总浓度可能看起来不低，但游离浓度可能明显低于总浓度。

本节通过改变蛋白结合率观察：

$$
C_{free}(t) = C_{total}(t) \times (1 - Protein\ Binding)
$$

蛋白结合率越高，游离浓度越低。

In [ ]:
def protein_binding_demo(
    total_concentration=20,
    mic=4
):
    binding_values = np.arange(0, 96, 5)
    free_values = total_concentration * (1 - binding_values / 100)

    fig, ax = plt.subplots()
    ax.plot(binding_values, free_values, marker="o", linewidth=2)
    ax.axhline(mic, linestyle="--", label=f"MIC = {mic:.1f} mg/L")
    ax.set_title("Effect of Protein Binding on Free Concentration")
    ax.set_xlabel("Protein binding (%)")
    ax.set_ylabel("Free concentration (mg/L)")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Protein binding (%)": binding_values,
        "Free concentration (mg/L)": np.round(free_values, 2),
        "Free concentration > MIC": ["Yes" if x > mic else "No" for x in free_values]
    })

    display(summary)


interact(
    protein_binding_demo,
    total_concentration=FloatSlider(value=20, min=1, max=100, step=1, description="Total C"),
    mic=FloatSlider(value=4, min=0.25, max=32, step=0.25, description="MIC")
);

interactive(children=(FloatSlider(value=20.0, description='Total C', min=1.0, step=1.0), FloatSlider(value=4.0…

## 13. 观察任务 4：为什么要看游离浓度？

请使用上面的蛋白结合率模拟完成以下任务。

### 任务 A：低蛋白结合率

设置：

- Total concentration = 20 mg/L
- MIC = 4 mg/L
- Protein binding 观察 0%–20%

思考：

- 游离浓度是否接近总浓度？
- 是否容易高于 MIC？

### 任务 B：高蛋白结合率

观察蛋白结合率 80%–95% 的情况。

思考：

- 总浓度相同的情况下，游离浓度是否明显下降？
- 如果只看总浓度，是否可能高估有效暴露？

### 任务 C：临床联系

思考：

> 低白蛋白血症、危重症或肾功能变化可能如何影响高蛋白结合率 β-内酰胺类药物的游离浓度和清除？

## 14. 本 Notebook 中模型的适用范围和局限性

本 Notebook 用于教学，不是临床处方系统。

本模拟做了以下简化：

1. 使用一室模型描述所有 β-内酰胺类药物。
2. 假设 CL 和 Vd 在模拟期间保持不变。
3. 假设药物按一级消除。
4. 使用固定蛋白结合率计算游离浓度。
5. 使用血浆浓度代替感染部位浓度。
6. 使用 \%fT > MIC 作为主要 PD 指标。
7. 未纳入真实患者病情、感染部位、病原菌负荷、免疫状态和药物毒性。

真实临床中，β-内酰胺类给药方案需要结合：

- 药品说明书
- 感染诊疗指南
- 药敏结果
- MIC
- 肾功能
- 感染部位
- 患者病情严重程度
- 药物稳定性和输注可行性
- 必要时的治疗药物监测

## 15. 自测题：β-内酰胺类 PK/PD

请根据本 Notebook 的内容完成以下自测题。建议先独立作答，再查看下一单元格中的参考答案。

---

### 题目 1：β-内酰胺类抗菌药物最常用的 PK/PD 指标是哪一个？

A. Cmax / MIC  
B. AUC / MIC  
C. %fT > MIC  
D. Tmax / MIC  

---

### 题目 2：为什么计算 β-内酰胺类 PK/PD 时通常关注游离浓度？

A. 因为蛋白结合药物通常不能直接发挥抗菌活性  
B. 因为总浓度永远等于游离浓度  
C. 因为蛋白结合率越高，游离浓度一定越高  
D. 因为 MIC 只适用于口服药物  

---

### 题目 3：在总日剂量相同的情况下，延长输注可能带来什么变化？

A. 一定使 AUC 变为 0  
B. 可能延长游离浓度高于 MIC 的时间  
C. 一定使 MIC 降低  
D. 一定使 CL 降低  

---

### 题目 4：同一给药方案下，MIC 升高通常会导致什么结果？

A. %fT > MIC 升高  
B. %fT > MIC 降低  
C. 蛋白结合率变为 0  
D. 药物清除率自动降低  

---

### 题目 5：CL 增加时，β-内酰胺类药物的浓度曲线最可能如何变化？

A. 药物消除变慢，浓度维持更久  
B. 药物消除变快，%fT > MIC 可能下降  
C. MIC 一定下降  
D. Vd 一定变为 0

## 16. 自测题参考答案

### 题目 1

**参考答案：C**

**解析：**  
β-内酰胺类通常表现为时间依赖性抗菌作用，常用 PK/PD 指标是：

$$
\% fT > MIC
$$

即一个给药间隔内，游离药物浓度高于 MIC 的时间百分比。

---

### 题目 2

**参考答案：A**

**解析：**  
通常认为未与蛋白结合的游离药物部分更能直接分布并发挥抗菌活性。因此，计算 β-内酰胺类 PK/PD 指标时，需要关注：

$$
C_{free}(t) = C_{total}(t) \times (1 - Protein\ Binding)
$$

---

### 题目 3

**参考答案：B**

**解析：**  
延长输注可以让药物输入时间变长。虽然峰浓度可能较低，但游离浓度高于 MIC 的时间可能增加，因此可能提高 \%fT > MIC。

---

### 题目 4

**参考答案：B**

**解析：**  
MIC 越高，药物浓度越难超过 MIC。同一给药方案下，MIC 升高通常会使 \%fT > MIC 降低，可能导致原方案不再达标。

---

### 题目 5

**参考答案：B**

**解析：**  
CL 增加表示药物清除加快。在其他参数不变时，浓度下降更快，游离浓度高于 MIC 的时间可能缩短，因此 \%fT > MIC 可能下降。

## 17. 本 Notebook 小结

本 Notebook 通过一室静脉输注模型，介绍了 β-内酰胺类药物 PK/PD 的基本逻辑。

你应该掌握以下核心结论：

1. β-内酰胺类抗菌药物通常使用时间依赖性 PK/PD 指标进行评价。
2. 关键 PK/PD 指标是 $\%fT > MIC$，即游离药物浓度高于 MIC 的时间比例。
3. 与总浓度相比，游离药物浓度更直接反映抗菌活性。
4. MIC 越高，同一给药方案越难达到 PK/PD 目标。
5. 清除率升高会降低药物暴露，并可能降低 $\%fT > MIC$。
6. 延长输注或持续输注可能提高目标达成率，但仍需考虑临床可行性和药物稳定性。

本节的完整逻辑可以概括为：

$$
Dose + Infusion\ Strategy + CL + V_d \rightarrow C_{free}(t) \rightarrow \%fT > MIC \rightarrow PK/PD\ Target\ Attainment
$$

下一节将进一步学习：

> 氨基糖苷类 PK/PD 模拟：Cmax/MIC、峰浓度与谷浓度。

## 18. 主要参考资料

1. Osthoff M, Siegemund M, Balestra G, Abdul-Aziz MH, Roberts JA. **Prolonged administration of β-lactam antibiotics – a comprehensive review and critical appraisal.** Swiss Medical Weekly. 2016;146:w14368. DOI: 10.4414/smw.2016.14368.
2. DailyMed. **Meropenem for Injection, powder, for solution.** U.S. National Library of Medicine. 说明书中包括美罗培南静脉输注、蛋白结合率和药代动力学信息。
3. Infectious Diseases Society of America. **IDSA 2024 Guidance on the Treatment of Antimicrobial Resistant Gram-Negative Infections.** 该指南讨论了部分场景下延长输注 β-内酰胺类药物的应用。

本 Notebook 中的参数和情景用于教学演示，不应用于真实患者的个体化处方。